# Financial Analysis Notebook

This notebook turns the finance tracker transaction data into a compact analyst-style workflow using **pandas**, **matplotlib**, and **seaborn**.

## Questions Answered
- How do income and expenses move over time?
- Which categories consume the biggest share of spend?
- Is the savings rate improving or deteriorating?
- Where are month-over-month expense spikes happening?
- Which transactions look unusually large relative to normal behavior?
- Which categories are over or under budget?


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"

data_dir = Path("../data")
transactions = pd.read_csv(data_dir / "sample_transactions.csv", parse_dates=["transaction_date"])
budgets = pd.read_csv(data_dir / "sample_budgets.csv")

transactions["month"] = transactions["transaction_date"].dt.to_period("M").astype(str)
transactions["month_label"] = transactions["transaction_date"].dt.strftime("%b %Y")

expenses = transactions[transactions["type"] == "expense"].copy()
income = transactions[transactions["type"] == "income"].copy()

print("Transactions:", len(transactions))
print("Expense rows:", len(expenses))
print("Income rows:", len(income))
print("Months covered:", transactions["month"].nunique())
transactions.head()


## 1. Monthly Income vs Expense Trend

**Why it matters:** This is the clearest view of cash-flow health. If expenses trend upward faster than income, savings pressure usually follows.


In [ ]:
monthly_flows = (
    transactions.groupby(["month", "type"])["amount"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=["income", "expense"], fill_value=0)
    .reset_index()
)
monthly_flows["net_savings"] = monthly_flows["income"] - monthly_flows["expense"]
monthly_flows["month_dt"] = pd.to_datetime(monthly_flows["month"])

fig, ax = plt.subplots()
ax.plot(monthly_flows["month_dt"], monthly_flows["income"], marker="o", linewidth=2.5, label="Income")
ax.plot(monthly_flows["month_dt"], monthly_flows["expense"], marker="o", linewidth=2.5, label="Expense")
ax.fill_between(
    monthly_flows["month_dt"],
    monthly_flows["income"],
    monthly_flows["expense"],
    where=monthly_flows["income"] >= monthly_flows["expense"],
    color="#10b981",
    alpha=0.12,
    label="Positive spread",
)
ax.set_title("Monthly Income vs Expense Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Amount")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

monthly_flows[["month", "income", "expense", "net_savings"]]


**Business insight:** Months where the gap between income and expenses narrows are early signals of cash-flow compression and should trigger a budget review.


## 2. Category-Wise Spending Breakdown

**Why it matters:** Share-of-wallet analysis shows which categories dominate total spend and where cost reduction efforts would have the biggest payoff.


In [ ]:
category_breakdown = (
    expenses.groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index(name="total_spend")
)
category_breakdown["pct_of_total"] = (
    category_breakdown["total_spend"] / category_breakdown["total_spend"].sum() * 100
).round(2)

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(
    data=category_breakdown,
    y="category",
    x="pct_of_total",
    hue="category",
    dodge=False,
    legend=False,
    ax=ax,
)
ax.set_title("Category Share of Total Expense")
ax.set_xlabel("Percent of total expense")
ax.set_ylabel("Category")
for idx, value in enumerate(category_breakdown["pct_of_total"]):
    ax.text(value + 0.2, idx, f"{value:.1f}%", va="center")
plt.tight_layout()
plt.show()

category_breakdown


**Business insight:** High-share categories represent the best targets for cost optimization because even small percentage reductions can materially improve savings.


## 3. Savings Rate Over Time

**Why it matters:** Savings rate is one of the clearest personal-finance efficiency metrics. It normalizes savings by income, making month-to-month comparisons more meaningful.


In [ ]:
savings_rate = monthly_flows.copy()
savings_rate["savings_rate_pct"] = np.where(
    savings_rate["income"] > 0,
    (savings_rate["net_savings"] / savings_rate["income"]) * 100,
    np.nan,
).round(2)

fig, ax = plt.subplots()
sns.lineplot(data=savings_rate, x="month_dt", y="savings_rate_pct", marker="o", linewidth=2.5, ax=ax)
ax.set_title("Savings Rate Over Time")
ax.set_xlabel("Month")
ax.set_ylabel("Savings rate (%)")
ax.axhline(30, linestyle="--", linewidth=1.5, color="#f59e0b", label="30% reference")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

savings_rate[["month", "income", "expense", "net_savings", "savings_rate_pct"]]


**Business insight:** A declining savings rate often indicates lifestyle inflation or rising discretionary spending pressure, even if income remains stable.


## 4. Month-over-Month Growth Rate for Expenses

**Why it matters:** Growth-rate analysis flags acceleration in spending. It helps explain not just how much was spent, but how quickly the spending pattern is changing.


In [ ]:
expense_growth = monthly_flows[["month", "expense"]].copy()
expense_growth["prev_month_expense"] = expense_growth["expense"].shift(1)
expense_growth["mom_growth_pct"] = (
    (expense_growth["expense"] - expense_growth["prev_month_expense"])
    / expense_growth["prev_month_expense"]
    * 100
).round(2)

fig, ax = plt.subplots()
sns.barplot(data=expense_growth, x="month", y="mom_growth_pct", color="#6366f1", ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Month-over-Month Growth in Expenses")
ax.set_xlabel("Month")
ax.set_ylabel("Growth rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

expense_growth


**Business insight:** Sudden MoM spikes often reveal seasonal behavior, large one-off purchases, or deteriorating spend discipline that should be investigated quickly.


## 5. Outlier / Anomaly Detection

**Why it matters:** Large abnormal transactions can distort the monthly picture. Detecting them helps separate recurring behavior from exceptional events.


In [ ]:
category_stats = expenses.groupby("category")["amount"].agg(["mean", "std"]).reset_index()
anomalies = expenses.merge(category_stats, on="category", how="left")
anomalies["std"] = anomalies["std"].replace(0, np.nan)
anomalies["z_score"] = ((anomalies["amount"] - anomalies["mean"]) / anomalies["std"]).round(2)
anomaly_table = (
    anomalies[anomalies["z_score"].abs() >= 2]
    .sort_values("z_score", ascending=False)
    [["transaction_date", "category", "title", "amount", "description", "z_score"]]
)

print("Anomalies flagged:", len(anomaly_table))
anomaly_table.head(15)


**Business insight:** Outliers should be reviewed before they are treated as a normal baseline, otherwise they can inflate budgets and distort trend models.


## 6. Budget vs Actual Variance by Category

**Why it matters:** Variance analysis shows whether overspending is broad-based or concentrated in a few categories. This is one of the most actionable management views in finance reporting.


In [ ]:
observed_months = expenses["month"].nunique()
actual_by_category = expenses.groupby("category")["amount"].sum().reset_index(name="actual_spend")
budget_variance = actual_by_category.merge(
    budgets[["category", "limit_amount"]],
    on="category",
    how="left",
)
budget_variance["annualized_budget"] = budget_variance["limit_amount"] * observed_months
budget_variance["variance_amount"] = budget_variance["actual_spend"] - budget_variance["annualized_budget"]
budget_variance["variance_pct"] = np.where(
    budget_variance["annualized_budget"] > 0,
    (budget_variance["variance_amount"] / budget_variance["annualized_budget"]) * 100,
    np.nan,
).round(2)
budget_variance = budget_variance.sort_values("variance_pct", ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
compare_df = budget_variance.melt(
    id_vars="category",
    value_vars=["actual_spend", "annualized_budget"],
    var_name="series",
    value_name="amount",
)
sns.barplot(data=compare_df, x="category", y="amount", hue="series", ax=ax)
ax.set_title("Budget vs Actual Spend by Category")
ax.set_xlabel("Category")
ax.set_ylabel("Amount")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

budget_variance[
    ["category", "actual_spend", "annualized_budget", "variance_amount", "variance_pct"]
]


**Business insight:** Categories with persistent positive variance are the best candidates for tighter controls, budget resets, or behavioral interventions.


## Key Takeaways

- The notebook covers monthly cash flow, spending mix, savings efficiency, expense acceleration, anomaly detection, and budget variance in one finance-analyst workflow.
- Each section is designed to answer a portfolio-friendly business question rather than only produce a chart.
- The outputs can be translated directly into dashboard metrics, reporting views, and recommendation logic for the app.
